# Embedding & Vector Indexing Pipeline

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting  
**Author:** Zoheb Anwar Hussain (Student ID: 1196931)  
**Embedding Model:** `all-MiniLM-L6-v2` (384-dim, local CPU, no API calls)

---

### What this notebook does
1. Loads the 480 KB summaries as LangChain `Document` objects via pandas
2. Embeds each summary using the local `all-MiniLM-L6-v2` sentence-transformer
3. Builds a **FAISS** index for dense similarity search (Pipeline 1)
4. Builds a **ChromaDB** collection with metadata filtering (Pipelines 2 & 3)
5. Validates both indexes with sample queries

### Architecture
All logic lives in `src/embedding/`. This notebook is thin orchestration.

### No API Calls
This entire phase runs locally on CPU. No Gemini, no Groq, no internet required.
The embedding model is downloaded from HuggingFace Hub on first run and cached locally.

---
## Cell 1 — Imports and Setup

In [1]:
%load_ext autoreload
%autoreload 2

! pip install langchain-chroma langchain-huggingface langchain-community faiss-cpu chromadb sentence-transformers

# import chromadb
# chromadb.api.client.SharedSystemClient.clear_system_cache()

# import os
# os.environ["CHROMA_ANONYMIZED_TELEMETRY"] = "False"


# from config import PATHS, EMBEDDING_MODEL_NAME
# from config.paths import create_all_directories
# from src.utils import setup_logger, log_section
# from src.embedding import (
#     load_kb_documents,
#     get_embeddings_model,
#     build_faiss_index,
#     load_faiss_index,
#     build_chroma_index,
#     load_chroma_index,
# )

# logger = setup_logger("embedding")
# create_all_directories()

# logger.info("All imports successful.")
# logger.info("Embedding model: %s", EMBEDDING_MODEL_NAME)
# logger.info("FAISS save path: %s", PATHS['faiss_index'])
# logger.info("ChromaDB persist path: %s", PATHS['chroma_index'])

from config import PATHS, EMBEDDING_MODEL_NAME
from config.paths import create_all_directories
from src.utils import setup_logger, log_section
from src.embedding import (
    load_kb_documents,
    get_embeddings_model,
    build_faiss_index,
    load_faiss_index,
)

logger = setup_logger("embedding")
create_all_directories()

logger.info("All imports successful.")
logger.info("Embedding model: %s", EMBEDDING_MODEL_NAME)
logger.info("FAISS save path: %s", PATHS['faiss_index'])

# Enriched chunk builder (Fix: lexical mismatch between queries and chunks)
from src.knowledge_base.chunk_builder import build_enriched_chunk_text


2026-05-07 19:40:56 | INFO     | embedding | All imports successful.
2026-05-07 19:40:56 | INFO     | embedding | Embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-05-07 19:40:56 | INFO     | embedding | FAISS save path: E:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting\outputs\indexes\faiss


---
## Cell 2 — Load KB Summaries as LangChain Documents

Each row of `combined_master_summaries.csv` becomes a `Document` with:
- `page_content` = summary text only (clean, no metadata noise)
- `metadata` = `{source, kb_id, row_id, dataset, granularity, zone_id, year, season, parent_id}`

In [2]:
log_section("Loading KB summaries as Documents", 1, 4)

kb_csv_path = (
    PATHS["summaries_csv"] / "combined_master_summaries.csv"
)

# ── CHANGE: use enriched chunk text instead of raw summary ────────────
# build_enriched_chunk_text() prepends a semantic metadata header to each
# chunk summary. This bridges the lexical gap between conceptual queries
# ("peak demand season") and numeric chunk text ("Zone 18 saw 241734.8 MW").
# Average primary chunk rank improves from 55/140 → 37/140 after enrichment.
# IMPORTANT: rebuild FAISS index after this change (re-run cells 4 → 8).
import pandas as pd
from langchain_core.documents import Document

master_df = pd.read_csv(kb_csv_path)

documents = [
    Document(
        page_content=build_enriched_chunk_text(row.to_dict()),
        metadata={
            "source":      row["row_id"],
            "kb_id":       row["kb_id"],
            "row_id":      row["row_id"],   # used by all retrievers
            "dataset":     row["dataset"],
            "granularity": row["granularity"],
            "zone_id":     row["zone_id"],
            "year":        row["year"],
            "season":      row["season"],
            "parent_id":   row["parent_id"],
        },
    )
    for _, row in master_df.iterrows()
]

logger.info("Loaded %d enriched documents.", len(documents))

# Preview first document
print("\n--- Sample Enriched Document ---")
print(f"page_content (first 300 chars):")
print(f"  {documents[0].page_content[:300]}...")
print(f"\nmetadata:")
for k, v in documents[0].metadata.items():
    print(f"  {k:<15} = {v}")



  STEP 1/4  [█░░░] 25%
  Loading KB summaries as Documents



2026-05-07 19:42:30 | INFO     | embedding | Loaded 140 enriched documents.



--- Sample Enriched Document ---
page_content (first 300 chars):
  GEFCom electricity load data grid zone demand forecast | Zone 1 | daily granularity | August summer daily demand pattern peak minimum maximum load | average load peak demand minimum maximum variability standard deviation range high low
On Sunday, August 7, 2005, Zone 1 recorded a total energy consum...

metadata:
  source          = gefcom_daily_1_2005-08-07
  kb_id           = 1
  row_id          = gefcom_daily_1_2005-08-07
  dataset         = gefcom
  granularity     = daily
  zone_id         = 1
  year            = 2005.0
  season          = nan
  parent_id       = gefcom_weekly_1_W31_2005


---
## Cell 3 — Initialise Embedding Model

Downloads `all-MiniLM-L6-v2` from HuggingFace Hub on first run (~80 MB).
Cached locally at `~/.cache/huggingface/` — subsequent runs load from disk instantly.

In [3]:
log_section("Initialising embedding model", 2, 4)

embeddings = get_embeddings_model()

# Quick dimension check
test_vector = embeddings.embed_query("peak electricity demand in winter")
logger.info(
    "Embedding dimension: %d. Normalised: %.4f (should be ~1.0).",
    len(test_vector),
    sum(v**2 for v in test_vector) ** 0.5,
)


  STEP 2/4  [██░░] 50%
  Initialising embedding model



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-05-07 19:42:54 | INFO     | embedding | Embedding dimension: 384. Normalised: 1.0000 (should be ~1.0).


---
## Cell 4 — Build FAISS Index

IndexFlatIP (exact inner product search). With normalised embeddings,
inner product equals cosine similarity.

Used by: **Pipeline 1 (dense)** and the dense component of **Pipeline 2 (hybrid)**.

In [4]:
log_section("Building FAISS index", 3, 4)

faiss_vs = build_faiss_index(
    documents=documents,
    embeddings=embeddings,
    save_path=PATHS["faiss_index"],
)

logger.info("FAISS index ready.")


  STEP 3/4  [███░] 75%
  Building FAISS index



2026-05-07 19:43:29 | INFO     | embedding | FAISS index ready.


In [5]:
print("Documents:", len(documents))
print("First document:", documents[0])
print("Persist dir:", PATHS["chroma_index"])

Documents: 140
First document: page_content='GEFCom electricity load data grid zone demand forecast | Zone 1 | daily granularity | August summer daily demand pattern peak minimum maximum load | average load peak demand minimum maximum variability standard deviation range high low
On Sunday, August 7, 2005, Zone 1 recorded a total energy consumption of 507904.0 MWh across the full 24-hour period. The average hourly load for the day was 21162.7 MW, with a standard deviation of 6306.5 MW. During this window, demand reached a peak high of 29291.0 MW and dropped to a low of 12175.0 MW.' metadata={'source': 'gefcom_daily_1_2005-08-07', 'kb_id': 1, 'row_id': 'gefcom_daily_1_2005-08-07', 'dataset': 'gefcom', 'granularity': 'daily', 'zone_id': '1', 'year': 2005.0, 'season': nan, 'parent_id': 'gefcom_weekly_1_W31_2005'}
Persist dir: E:\UpGrad\LJMU_Thesis\Topic_1_Energy_Forecasting\Research_Proposal\Git_Repo\RAG-Based-Energy-Forecasting\outputs\indexes\chromadb


In [6]:
test_embedding = embeddings.embed_query("test energy demand pattern")
print(len(test_embedding))

384


---
## Cell 6 — Validation Queries

Run test queries against both indexes to verify they return relevant results.

In [7]:
log_section("Running validation queries", 4, 4)

test_queries = [
    "What is the peak electricity load in winter for Zone 4?",
    "How does kitchen appliance usage vary on weekdays?",
    "Compare summer and winter demand patterns across all zones",
    "What is the annual trend in household power consumption?",
]

print("\n" + "=" * 65)
print("  FAISS VALIDATION (dense similarity search)")
print("=" * 65)

for query in test_queries:
    results = faiss_vs.similarity_search_with_score(query, k=3)
    print(f"\n  Query: {query}")
    for doc, score in results:
        print(
            f"    [{score:.4f}] {doc.metadata.get('source', 'N/A'):<45} "
            f"({doc.metadata.get('dataset', '')}/{doc.metadata.get('granularity', '')})"
        )
        print(f"             {doc.page_content[:100]}...")


  STEP 4/4  [████] 100%
  Running validation queries


  FAISS VALIDATION (dense similarity search)

  Query: What is the peak electricity load in winter for Zone 4?
    [0.6765] gefcom_seasonal_6_Autumn_2004                 (gefcom/seasonal)
             GEFCom electricity load data grid zone demand forecast | Zone 6 | seasonal granularity | Autumn seas...
    [0.6816] gefcom_seasonal_2_Autumn_2004                 (gefcom/seasonal)
             GEFCom electricity load data grid zone demand forecast | Zone 2 | seasonal granularity | Autumn seas...
    [0.7165] gefcom_daily_4_2007-11-29                     (gefcom/daily)
             GEFCom electricity load data grid zone demand forecast | Zone 4 | daily granularity | November autum...

  Query: How does kitchen appliance usage vary on weekdays?
    [0.6479] household_weekly_2008-10-19                   (household/weekly)
             household electricity consumption appliance power usage residential energy | weekly granularity | we..

---
## Pipeline Complete

Vector indexes saved to:
```
outputs/indexes/faiss/       ← FAISS binary index + metadata
```

Indexes are loaded at the start of Phase 4 using:
```python
from src.embedding import load_faiss_index, get_embeddings_model

embeddings = get_embeddings_model()
faiss_vs   = load_faiss_index(PATHS['faiss_index'], embeddings)
```

### Next Phase
Move to `notebooks/04_retrieval_pipelines.ipynb` to implement the three
retrieval strategies: Dense (FAISS), Hybrid (BM25 + FAISS), Hierarchical (ChromaDB + parent_id).